# Calculating a Shock Extreme Typical Meteorological Year -- Methodology Walkthrough

Extreme meteorological years (XMYs) are intended to capture extreme events, like heat waves and cold snaps. We define two types of XMYs:

1. persistence: sustained extremes (ex: an unusualy hot summer)
2. shock: system shocks, weather events (ex: heat waves, cold snaps)

This notebook covers to the steps to generate shock XMYs.

The shock XMY methodology utilizes the following small changes to the TMY methodology to get at cold and hot extremes:
1. Modifies the Finkelstein-Schafer statistic calculation to use the relative, rather than absolute, difference. This preserves the direction of the difference.
2. When selecting the candidate months, select values that differ MOST from the climatological CDF in the negative direction for cold shock events, and the positive direction for hot shock events.

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import hvplot.xarray
from importlib.resources import files
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

import climakitae as ck
from climakitae.core.constants import UNSET
from climakitae.core.data_export import write_tmy_file

# Initialize ClimateData object
cd = ck.ClimateData(verbosity=-1)

### Step 1: Grab and process all required input data
We use the TMY3 method to guide variable selection. The method selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance.

#### Step 1a: Select location of interest
XMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the XMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too.

In [ ]:
# read in station file of CA HadISD stations
stn_filepath_s3 = "s3://cadcat/hadisd/hadisd_stations.csv"
stn_file = pd.read_csv(stn_filepath_s3, index_col=[0])
stn_file.head(5)

In [ ]:
# grab airport
stn_name = "Ontario International Airport (KONT)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()

# Output results
print("station coordinates:", (stn_lat, stn_lon))
print("station code:", stn_code)

Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [ ]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating a XMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data; we will use 30 years to represent a standard climatological period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

We will also process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">1990-2020 period</span> as an example. **Note**: An additional year (2021) is also loaded to pad the end of the dataset after converting to local time in the next few steps -- this is done for you when subsetting for the data. 

In [ ]:
# selected reference period
start_year = 2005
end_year = 2020

#### Step 1c: Retrieve variables in catalog
It is important to note that not all models in the Cal-Adapt: Analytics Engine have the solar variables critical for XMY file generation - in fact, only 4 do! We will carefully subset our variables to ensure that the same 4 models are selected for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` function corrects for this, and ensures that all data are within the same daily timestamp.

In [ ]:
# selected models
data_models = [
    "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
    "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
]

# map variable names to descriptive names and units
var_info = {
    "t2": {"long_name": "Air temperature at 2m", "units": "degC"},
}

Now that we have set up the model list, let's start retrieving data! We will need to calculate summary statistics and reduce to daily timescales for the following variables:

In [ ]:
t2_ds = (
    cd.catalog("cadcat")
    .variable("t2")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "convert_units": "degC",
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
t2_ds = t2_ds.sel(sim=data_models, time=slice(str(start_year), str(end_year))).t2

# Load into memory using dask progress bar
with ProgressBar():
    t2_ds.load()

In [ ]:
# max air temp
max_airtemp_data = t2_ds.resample(time="1D").max()  # daily max air temp
max_airtemp_data.name = "Daily max air temperature"  # rename for clarity

# min air temp
min_airtemp_data = t2_ds.resample(time="1D").min()  # daily min air temp
min_airtemp_data.name = "Daily min air temperature"  # rename for clarity

# mean air temp
mean_airtemp_data = t2_ds.resample(time="1D").mean()  # daily mean air temp
mean_airtemp_data.name = "Daily mean air temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del t2_ds

Retrieve and calculate max and mean wind speed:

In [ ]:
# wind speed
ws_ds = (
    cd.catalog("cadcat")
    .variable("wind_speed_10m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
ws_ds = ws_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).wind_speed_10m

# Load into memory using dask progress bar
with ProgressBar():
    ws_ds.load()

In [ ]:
# max wind speed
max_windspd_data = ws_ds.resample(time="1D").max()  # daily max wind speed
max_windspd_data.name = "Daily max wind speed"  # rename for clarity

# mean wind speed
mean_windspd_data = ws_ds.resample(time="1D").mean()  # daily mean wind speed
mean_windspd_data.name = "Daily mean wind speed"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del ws_ds

Retrieve and calculate max, min, and mean dew point temperature:

In [ ]:
# dew point temperature
dewpt_ds = (
    cd.catalog("cadcat")
    .variable("dew_point_2m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_units": "degC",
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
dewpt_ds = dewpt_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).dew_point_2m

# Load into memory using dask progress bar
with ProgressBar():
    dewpt_ds.load()

In [ ]:
# max dew point
max_dewpt_data = dewpt_ds.resample(time="1D").max()  # daily max dewpoint temp
max_dewpt_data.name = "Daily max dewpoint temperature"  # rename for clarity

# min dew point
min_dewpt_data = dewpt_ds.resample(time="1D").min()  # daily min dewpoint temp
min_dewpt_data.name = "Daily min dewpoint temperature"  # rename for clarity

# mean dew point
mean_dewpt_data = dewpt_ds.resample(time="1D").mean()  # daily mean dewpoint temp
mean_dewpt_data.name = "Daily mean dewpoint temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del dewpt_ds

Next, retrieve global horizontal irradiance. GHI is within the Analytics Engine catalog at daily resolutions, but for the XMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. The same process is repeated for the direct normal irradiance. 

In [ ]:
# global irradiance
global_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swdnb")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
global_irradiance_ds = global_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swdnb

# Load into memory using dask progress bar
with ProgressBar():
    global_irradiance_ds.load()

In [ ]:
# total global horizontal irradiance (accumulation of hourly data over a 24-hour period)
total_ghi_data = global_irradiance_ds.resample(time="1D").sum()
total_ghi_data.name = "Global horizontal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del global_irradiance_ds

The same process is repeated for the direct normal irradiance.

In [ ]:
# direct normal irradiance
direct_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swddni")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
direct_irradiance_ds = direct_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swddni

# Load into memory using dask progress bar
with ProgressBar():
    direct_irradiance_ds.load()

In [ ]:
# total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_data = direct_irradiance_ds.resample(time="1D").sum()
total_dni_data.name = "Direct normal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del direct_irradiance_ds

#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now.

In [ ]:
all_vars = xr.merge(
    [
        max_airtemp_data,
        min_airtemp_data,
        mean_airtemp_data,
        max_dewpt_data,
        min_dewpt_data,
        mean_dewpt_data,
        max_windspd_data,
        mean_windspd_data,
        total_ghi_data,
        total_dni_data,
    ]
)

### Step 2: Calculate cumulative distribution functions

The cumulative distribution function gives the proportion of values that are less than or equal to a specified value of the index. In this case, we want to identify months that differ most from the long-term climatology for each variable as possible, indicating months that are "extreme".  

#### Step 2a: Calculate long-term climatology CDFs for each index
First, we need to calculate the long-term climatology for each index for each month so we can establish the baseline pattern. 

In [ ]:
# load saved all_vars eagerly
all_vars = xr.load_dataset("all_vars.nc")

In [ ]:
def compute_cdf(da):
    """Compute the cumulative density function for an input DataArray"""
    da_np = da.values  # Get numpy array of values
    num_samples = 1024  # Number of samples to generate
    count, bins_count = np.histogram(  # Create a numpy histogram of the values
        da_np,
        bins=np.linspace(
            da_np.min(),  # Start at the minimum value of the array
            da_np.max(),  # End at the maximum value of the array
            num_samples,
        ),
    )
    cdf_np = np.cumsum(count / sum(count))  # Compute the CDF

    # Turn the CDF array into xarray DataArray
    # New dimension is the bin values
    cdf_da = xr.DataArray(
        [bins_count[1:], cdf_np],
        dims=["data", "bin_number"],
        coords={
            "data": ["bins", "probability"],
        },
    )
    cdf_da.name = da.name
    return cdf_da


def get_cdf_by_sim(da):
    # Group the DataArray by simulation
    return da.groupby("sim").map(compute_cdf)


def get_cdf_by_mon_and_sim(da):
    # Group the DataArray by month in the year
    return da.groupby("time.month").map(get_cdf_by_sim)


def get_cdf(ds):
    """Get the cumulative density function.

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """
    return ds.map(get_cdf_by_mon_and_sim)

In [ ]:
cdf_climatology = get_cdf(all_vars)

In the plot below, we'll display maximum air temperature to assess the climatological CDF pattern, but you can modify the variable here to one of your choosing to see the pattern too! Also select a different month by moving the slider bar to see the pattern throughout the year.

In [ ]:
def plot_cdf(cdf_da: xr.DataArray) -> hvplot:
    """
    Plot CDF with simulations as a grouped overlay.

    Parameters
    ----------
    cdf_da : xr.DataArray
        Output of compute_cdf, with a 'data' dimension containing 'bins' and 'probability'.

    Returns
    -------
    hvplot
        CDF plot object.
    """
    ds = xr.merge(
        [
            cdf_da.sel(data="probability", drop=True).rename("probability"),
            cdf_da.sel(data="bins", drop=True).rename("bins"),
        ]
    )
    cdf_pl = ds.hvplot(
        "bins",
        "probability",
        by="sim",
        widget_location="bottom",
        grid=True,
        xlabel=f"{cdf_da.name} ({cdf_da.attrs['units']})",
        xlim=(ds["bins"].min().item(), ds["bins"].max().item()),
        ylabel="Probability (0-1)",
    )
    return cdf_pl

In [ ]:
# Choose your desired variable
var = "Daily mean air temperature"

# Make plot
plot_cdf(cdf_climatology[var])

#### Step 2b: Calculate CDFs for each index for all months

Next, we will calculate CDF for each month and each variable, for which we ultimately will compare against climatology. For the individual months, we must also exclude the period of time during a major volcanic eruption (Pinatubo: June 1991 to December 1994) as the aerosols have an impact on solar variables. The cells below functions exclude this data from our data next. 

In [ ]:
def get_cdf_monthly(ds):
    """Get the cumulative density function by unique mon-yr combos

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """

    def get_cdf_mon_yr(da):
        return da.groupby("time.year").apply(get_cdf_by_mon_and_sim)

    return ds.apply(get_cdf_mon_yr)

In [ ]:
cdf_monthly = get_cdf_monthly(all_vars)

Now we'll remove the volcanic years: 

In [ ]:
# Remove the years for the Pinatubo eruption
cdf_monthly = cdf_monthly.where(
    (~cdf_monthly.year.isin([1991, 1992, 1993, 1994])), np.nan, drop=True
)

Like the climatology CDF figure above, let's check out the individual months next. You can modify the variable, and month-year to display too.

In [ ]:
# Choose your desired variable
var = "Daily mean air temperature"

# Make the plot
plot_cdf(cdf_monthly[var])

In [ ]:
## we have heard from utilities that air temperature is the single priority variable they need from XMYs
## instead of weighting to identify which year is selected
## we prioritize based on air temperature extremes

extreme = 'cold'
if extreme == "cold":
    var = "Daily min air temperature" # proceed with minimum air temperature for cold shock
elif extreme == "hot":
    var = "Daily max air temperature" # proceed with meximum air temperature for hot shock

miroc_cdf = cdf_climatology.sel(sim="wrf_ucla_miroc6_ssp370_r1i1p1f1")
miroc_cdf_jan = miroc_cdf.sel(month=1)
miroc_cdf_jan = miroc_cdf_jan[var]
miroc_cdf_jan

miroc_mcdf_jan = cdf_monthly.sel(sim="wrf_ucla_miroc6_ssp370_r1i1p1f1")
miroc_mcdf_jan = miroc_mcdf_jan.sel(month=1)
miroc_mcdf_jan = miroc_mcdf_jan[var]
miroc_mcdf_jan


# PLOT
fig, ax = plt.subplots(figsize=(12, 8))

# mean
c1 = miroc_cdf_jan
ds1 = xr.merge(
    [
        c1.sel(data="probability", drop=True).rename("probability"),
        c1.sel(data="bins", drop=True).rename("bins"),
    ]
)
ax.plot(
    ds1["bins"].values,
    ds1["probability"].values,
    linewidth=2, 
    color='k',
)

# each year
c2 = miroc_mcdf_jan
ds2 = xr.merge(
    [
        c2.sel(data="probability", drop=True).rename("probability"),
        c2.sel(data="bins", drop=True).rename("bins"),
    ]
)
x = ds2['bins']
for yr in ds2["year"].values:
    y = ds2["probability"].sel(year=yr)
    x = ds2["bins"].sel(year=yr)

    ax.plot(
        x.values,
        y.values,
        alpha=0.5,
        linewidth=1,
        label=str(yr),
    )

ax.set_xlabel(f"{c1.name} ({c1.attrs['units']})")
ax.set_ylabel("Probability (0-1)")
ax.set_xlim(ds2["bins"].min().item(), ds2["bins"].max().item())
ax.legend(loc='upper left')

ax.grid(True)

### Step 3: Select candidate months for consideration

Now that we have the distributions for the long-term climatology of our 30-year period, and the corresponding distribution for each month in that 30-year period, we need to assess how each individual month compares to the long-term climatology. In essence, we are looking for the individual months that most differ from the climatology distribution in the desired direction.

In [ ]:
def find_hot_cold_extreme_from_median(
    cold_year: xr.DataArray,
    cold_clim: xr.DataArray,
    target: float = 0.5,
    extreme: str = "cold",  # "cold" or "hot"
):
    """
    Identify hottest or coldest year based on deviation of the median (CDF=0.5)
    temperature value across years.

    Parameters
    ----------
    cold_year : xr.DataArray
        dims: (year, data, bin_number)

    cold_clim : xr.DataArray
        climatology with same structure

    target : float
        CDF level (default 0.5)

    extreme : str
        "cold" -> pick minimum deviation
        "hot"  -> pick maximum deviation

    Returns
    -------
    results : list
        median values per year

    worst_year : scalar
        identified extreme year
    """

    # -----------------------------
    # Step 1: climatology median (p50 reference)
    # -----------------------------
    clim_prob = cold_clim.sel(data="probability")
    clim_bins = cold_clim.sel(data="bins")

    clim_idx = np.abs(clim_prob - target).argmin(dim="bin_number")
    clim_05 = clim_bins.isel(bin_number=clim_idx).values

    # -----------------------------
    # Step 2: per-year median extraction
    # -----------------------------
    yr_prob = cold_year.sel(data="probability")
    yr_bins = cold_year.sel(data="bins")

    idx_list = (
        np.abs(yr_prob - target)
        .argmin(dim="bin_number")
        .values
        .tolist()
    )

    results = []

    years = cold_year["year"].values

    for i, yr in enumerate(years):
        idx = idx_list[i]

        val = yr_bins.sel(year=yr).isel(bin_number=idx).values
        results.append(val)

    results = np.array(results)

    # -----------------------------
    # Step 3: compute anomalies vs climatology
    # -----------------------------
    anomaly = results - clim_05

    # -----------------------------
    # Step 4: pick extreme year
    # -----------------------------
    if extreme == "cold":
        worst_idx = np.argmin(anomaly)   # most negative deviation
    elif extreme == "hot":
        worst_idx = np.argmax(anomaly)   # most positive deviation
    else:
        raise ValueError("extreme must be 'cold' or 'hot'")

    worst_year = years[worst_idx]

    return results, anomaly, worst_year

In [ ]:
def test(cdf_monthly, cdf_climatology,extreme):
    if extreme == "cold":
        var = "Daily min air temperature"
    elif extreme == "hot":
        var = "Daily max air temperature"
    
    subset_clim = cdf_climatology[var]
    subset_year = cdf_monthly[var]

    results = []
    for sim in cdf_monthly.sim.values:
        clim_sim  = subset_clim .sel(sim=sim)
        year_sim = subset_year.sel(sim=sim)
        for mon in cdf_monthly.month.values:
            clim_month  = clim_sim .sel(month=mon)
            year_month = year_sim.sel(month=mon)
            _, _, worst_year = find_hot_cold_extreme_from_median(year_month, clim_month, extreme=extreme) 
            row = {
            "month": mon,
            "sim": sim,
            "year": worst_year
        }
            results.append(row)
    top_df = pd.DataFrame(results)

    return top_df


In [ ]:
top_df = test(cdf_monthly, cdf_climatology,'cold')

### Step 4: Generate the shock XMY data outputs

Generally, the following data is outputted using the XMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
def generate_xmy_data(
    top_df: pd.DataFrame,
    start_year: int,
    end_year: int,
    lat: float,
    lon: float,
    data_models: list,
) -> dict:
    """Generate typical meteorological year data.

    Parameters
    ----------
    top_df : pd.DataFrame
        Table with columns month, simulation, and year. Each month-sim-yr
        combo represents the top candidate with the lowest weighted FS statistic.
    start_year : int
    end_year : int
    lat : float
    lon : float
    data_models : list

    Returns
    -------
    dict of str : pd.DataFrame
        Dictionary in the format {simulation: TMY DataFrame}.
    """
    # Define climate data object with minimal terminal output
    cd = ck.ClimateData(verbosity=-2)

    # Define variables and unit conversions
    var_info = {
        "t2": {
            "long_name": "Air temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "dew_point_2m": {
            "long_name": "Dew point temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "relative_humidity_2m": {
            "long_name": "Relative humidity (0-100)",
            "units": UNSET,
        },  # Use unit default (0-100)
        "swdnb": {
            "long_name": "Instantaneous downwelling shortwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddni": {
            "long_name": "Shortwave surface downward direct normal irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddif": {
            "long_name": "Shortwave surface downward diffuse irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "lwdnb": {
            "long_name": "Instantaneous downwelling longwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "wind_speed_10m": {
            "long_name": "Wind speed at 10m (m/s)",
            "units": UNSET,
        },  # Use unit default (m/s)
        "wind_direction_10m": {
            "long_name": "Wind direction at 10m (degrees)",
            "units": UNSET,
        },  # Use unit default (degrees)
        "psfc": {
            "long_name": "Surface pressure (Pa)",
            "units": UNSET,
        },  # Use unit default (Pa)
    }

    ## ================== GET DATA FROM CATALOG ==================
    print(
        f"STEP 1: RETRIEVING HOURLY DATA FROM CATALOG FOR {len(var_info)} VARIABLES\n"
    )
    all_vars_list = []
    for i, (var, info) in enumerate(var_info.items(), start=1):
        long_name = info["long_name"]
        units = info["units"]
        print(f"({i}/{len(var_info)}) Variable: {var}")
        print("Retrieving data from catalog...")
        data_by_var = (
            cd.catalog("cadcat")
            .variable(var)
            .activity_id("WRF")
            .institution_id("UCLA")
            .table_id("1hr")
            .grid_label("d03")
            .experiment_id(["historical", "ssp370"])
            .processes(
                {
                    "time_slice": (start_year, end_year + 1),
                    "clip": (lat, lon),
                    "convert_units": units,
                    "convert_to_local_time": {"convert": "yes"},
                }
            )
            .get()
        )

        # Subset simulations and convert to DataArray
        data_by_var = data_by_var.sel(
            sim=data_models, time=slice(str(start_year), str(end_year))
        )[var]

        # Update variable name to use long_name for clarity
        data_by_var.name = long_name

        print(f"Retrieved. Loading data into memory...")
        with ProgressBar():
            data_by_var.load()

        print(f"Data loaded into memory.\n")
        all_vars_list.append(data_by_var)

    # Merge data from all variables into a single xr.Dataset object
    # Drop unneeded coordinates
    all_vars_ds = xr.merge(all_vars_list)
    all_vars_ds = all_vars_ds.drop_vars(
        ["lakemask", "landmask", "x", "y", "Lambert_Conformal"], errors="ignore"
    )

    # Clear individual DataArrays from memory after merge
    del all_vars_list

    ## ================== CONSTRUCT TMY ==================
    print("\nSTEP 2: CALCULATING SHOCK EXTREME METEOROLOGICAL YEAR PER MODEL SIMULATION\n")
    tmy_df_all = {}
    for sim in all_vars_ds.sim.values:
        df_list = []
        print(f"Calculating TMY for simulation: {sim}")
        for mon in np.arange(1, 13, 1):
            # Get year corresponding to month and simulation combo
            year = top_df.loc[
                (top_df["month"] == mon) & (top_df["sim"] == sim)
            ].year.item()

            # Select data for unique month, year, and simulation
            data_at_stn_mon_sim_yr = all_vars_ds.sel(
                sim=sim,
                time=(all_vars_ds.time.dt.month == mon)
                & (all_vars_ds.time.dt.year == year),
            ).expand_dims("sim")

            # Reformat as dataframe
            df_by_mon_sim_yr = data_at_stn_mon_sim_yr.to_dataframe().reset_index()

            # Reformat time index to remove seconds
            df_by_mon_sim_yr["time"] = pd.to_datetime(
                df_by_mon_sim_yr["time"].values
            ).strftime("%Y-%m-%d %H:%M")

            df_list.append(df_by_mon_sim_yr)

        # Concatenate all DataFrames together
        tmy_df_by_sim = pd.concat(df_list)
        tmy_df_all[sim] = tmy_df_by_sim

        print("XMY PROCESS COMPLETE.")

    return tmy_df_all

In the next cell, we will run the `generate_xmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
xmy_data_to_export = generate_xmy_data(
    top_df, start_year, end_year, stn_lat, stn_lon, data_models
)

Let's observe what the XMY data looks like for one of the simulations:

In [ ]:
simulation = "wrf_ucla_miroc6_ssp370_r1i1p1f1"
xmy_data_to_export[simulation].head(5)

Next, we visualize the XMY data itself.

In [ ]:
# Make plot using pandas
ax = xmy_data_to_export[simulation].plot(
    x="time",
    y=[
        col
        for col in xmy_data_to_export[simulation]
        if col not in ["time", "sim", "lat", "lon"]
    ],
    subplots=True,
    figsize=(12, 12),
    legend=True,
    fontsize=10,
)

# Plot formatting
for a in ax.flatten():
    a.xaxis.label.set_fontsize(12)
    a.yaxis.label.set_fontsize(12)
    a.legend(loc="upper right", fontsize=14)
plt.suptitle(
    f"Typical Meteorological Year (simulation: {simulation})", fontsize=16, y=0.98
)
plt.tight_layout();

Lastly, let's export the XMY data below as csv files. There will be a file per simulation downloaded. When utilizing TMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional stn_elev argument to write_tmy_file; if no elevation value is provided, an elevation value of 0.0 is set as the default.

In [ ]:
for sim, xmy in xmy_data_to_export.items():
    filename = "temp_{0}_shock_xmy_{1}_{2}".format(
        extreme,stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        xmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="csv", #epw
    )